![](https://cascade.yonsei.ac.kr/workshop/rum26/rum26.jpg)

Hosted at Yonsei University, Seoul, South Korea (=Republic of Korea). A five‑day meeting for the RAMSES community and newcomers: code development, science highlights, tutorials, and collaborations.

 - Dates: Apr 30 (Thu)  2026

 - Venue: Yonsei University, Science Building B102



# **Nways GPU Profiling Workshop**

Welcome to the Nways GPU Profiling Workshop! 

As scientists and researchers, our main goal is to accelerate our applications using GPUs without getting lost in computer science jargon. During the previous labs, you worked on the Radial Distribution Function (RDF) code.

In this 2-hour hands-on session, we will not just run the code. We will look under the hood using **NVIDIA Profiling Tools (Nsight Systems & Nsight Compute)** to understand *why* our code is slow, *what* the compiler is doing, and *how* to verify our optimizations.

### **Step 0: Check the Environment**
First, let's verify our NVIDIA GPU, the installation paths of our tools (`ncu`, `nsys`), and the HPC SDK libraries.

In [ ]:
%%bash
echo "=================================================="
echo "                GPU & Driver Information          "
echo "=================================================="
nvidia-smi --query-gpu=gpu_name,compute_cap,driver_version --format=csv

echo -e "\n=================================================="
echo "             Tool Installation Paths              "
echo "=================================================="
echo "Compiler (nvc++): $(which nvc++)"
echo "Compiler (nvcc):  $(which nvcc)"
echo "Profiler (nsys):  $(which nsys)"
echo "Profiler (ncu):   $(which ncu)"

echo -e "\n=================================================="
echo "             CUDA Toolkit & Compilers             "
echo "=================================================="
nvcc --version | grep "release"
nvc++ --version | head -n 2

echo -e "\n=================================================="
echo "             CUDA Library Information             "
echo "=================================================="
echo "Checking key libraries in /opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64:"
ls -l /opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 | grep -E "libcudart|libnvToolsExt" | awk '{print $9}'

--- 
## **Part 1: Compiling and OpenACC Built-in Profiling**

Before we even run the advanced profiler, the compiler can tell us a lot about what it is doing with our GPU. 

### **1.1 Inspecting the Source Code**
We will be working with the file: `openacc/source_code/SOLUTION/rdf_parallel_directive.cpp`

Let's look at the core loop of our **Naive Parallel** OpenACC code. In this code, we only added simple `#pragma acc` directives to parallelize the loop, but we completely ignored memory transfers.

```cpp
#pragma acc parallel loop
for (int id1 = 0; id1 < numatm; id1++)
{
    #pragma acc loop
    for (int id2 = 0; id2 < numatm; id2++)
    {
        dx = d_x[frame * numatm + id1] - d_x[frame * numatm + id2];
        dy = d_y[frame * numatm + id1] - d_y[frame * numatm + id2];
        dz = d_z[frame * numatm + id1] - d_z[frame * numatm + id2];

        dx = dx - xbox * (round(dx / xbox));
        dy = dy - ybox * (round(dy / ybox));
        dz = dz - zbox * (round(dz / zbox));

        r = sqrtf(dx * dx + dy * dy + dz * dz);
        if (r < cut)
        // ... (Pair Calculation) ...
```

### **1.2 Compiling with Feedback (`-Minfo`)**
**Important Educational Lesson:** In C/C++, arrays (`d_x`, `d_y`, `d_z`) are passed as pointers (`double *`). The compiler doesn't know the size of these arrays. If we compile this with just `-acc`, it fails with a bug: `Accelerator restriction: size of the GPU copy is unknown`.

To bypass this bug and test our parallel loops, we tell the compiler to use **Unified Memory** by adding the `-gpu=managed` flag. The NVIDIA driver will automatically handle moving the memory for us!

In [ ]:
%%bash
cd openacc/source_code/SOLUTION

# Set the NVTX include path explicitly based on the HPC SDK 2024 structure
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Naive Parallel Version with Unified Memory (-gpu=managed)..."
nvc++ -O3 -w -I. ${NVTXLIB} -acc -gpu=cc89,managed -Minfo=accel -o rdf_parallel_c rdf_parallel_directive.cpp

**🔍 Check the Output:**
You should see `Generating NVIDIA GPU code`: The compiler successfully created a GPU kernel! Because of `-gpu=managed`, the compiler accepted the code even without knowing array sizes.

### **1.3 Executing the Code (Normal vs. Profile Mode)**
First, let's execute the code normally. *(Note: We execute the code from the `source_code` directory so it can find the dataset at `../../_common/input/alk.traj.dcd`)*

In [ ]:
%%bash
cd openacc/source_code

echo "Executing normally..."
./SOLUTION/rdf_parallel_c

The code runs and prints the output, but it doesn't tell us how the GPU performed. 

Before using heavy advanced profilers, the NVIDIA HPC SDK provides a fantastic built-in tool. By setting `NVCOMPILER_ACC_TIME=1` before execution, we get an immediate educational breakdown of kernel execution time and data transfer time. Let's try it!

In [ ]:
%%bash
cd openacc/source_code

echo "Executing with NVCOMPILER_ACC_TIME=1..."
NVCOMPILER_ACC_TIME=1 ./SOLUTION/rdf_parallel_c

**💡 What to look for:** At the very end of the execution output, you will now see an `Accelerator Kernel Timing data` table showing exactly how many microseconds were spent on the compute kernel versus Unified Memory page faults (`data copyin`, `data copyout`).

---
## **Part 2: Profiling with Nsight Systems (`nsys`)**

While the built-in time debug is great, **Nsight Systems (`nsys`)** creates a full visual timeline. By using the `--sample=cpu` and `--stats=true` flags, we can also see a `gprof`-like subroutine profile directly in the standard output.

### **2.1 Profile the Naive OpenACC Version**
We will profile the naive version we just compiled and let `nsys` print the full statistics to the screen.

In [ ]:
%%bash
cd openacc/source_code

echo "Profiling Naive Parallel Version (stdout is displayed!)..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --sample=cpu \
  --stats=true \
  --force-overwrite=true \
  -o profile_acc_parallel \
  ./SOLUTION/rdf_parallel_c

**🔍 Look at the Console Output:** Scroll up through the output of `nsys profile`. You will see tables for `NVTX Range Statistics` (showing the "Pair_Calculation" function), `CUDA Kernel Statistics`, and `CUDA Memory Operation Statistics`. Notice how much time is spent on Memory Operations (HtoD / DtoH)!

### **2.2 Inspecting the Data Optimized Code**
Let's look at `rdf_data_directive.cpp`. Notice how we explicitly manage data using `#pragma acc data copyin(...) copyout(...)`. Because we explicitly define the array sizes here, we no longer need the `-gpu=managed` flag!

In [ ]:
%%bash
cd openacc/source_code

echo "--- Inspecting Data Directives ---"
grep -B 2 -A 10 "#pragma acc data" SOLUTION/rdf_data_directive.cpp

### **2.3 Compiling the Data Optimized Version**

In [ ]:
%%bash
cd openacc/source_code/SOLUTION
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Data Optimized Version (No 'managed' flag needed!)..."
nvc++ -O3 -w -I. ${NVTXLIB} -acc -gpu=cc89 -Minfo=accel -o rdf_data_c rdf_data_directive.cpp

### **2.4 Profiling the Data Optimized Version**
Let's trace this optimized version. Pay attention to the `CUDA Memory Operation Statistics` table this time!

In [ ]:
%%bash
cd openacc/source_code

echo "Profiling Data Optimized Version..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_acc_data \
  ./SOLUTION/rdf_data_c

**💡 What changed?** Check the memory operations table in the output above. The overhead is significantly less, and the overall kernel execution time decreased dramatically!

---
## **Part 3: OpenMP Offload - Expanding Parallelism (Collapse)**

OpenMP is another popular model. A common optimization is **Loop Collapsing**. 
If you have two nested loops (e.g., `for i in 100`, `for j in 100`), they create $100 \times 100 = 10,000$ independent work items. Collapsing them allows the GPU to process all 10,000 threads simultaneously across its massive array of cores instead of managing them hierarchically.

### **3.1 Inspecting Basic OpenMP Offload Code**

In [ ]:
%%bash
cd openmp/source_code

echo "--- Basic OpenMP Target Region ---"
grep -A 5 "#pragma omp target teams distribute" SOLUTION/rdf_offload.cpp

### **3.2 Compiling & Profiling Basic OpenMP Offload**

In [ ]:
%%bash
cd openmp/source_code
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Basic OpenMP Offload..."
nvc++ -O3 -w -I. ${NVTXLIB} -mp=gpu -gpu=cc89 -o SOLUTION/rdf_omp_offload_c SOLUTION/rdf_offload.cpp

echo "Profiling Basic OpenMP Offload..."
nsys profile \
  --trace=cuda,openmp,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_omp_offload \
  ./SOLUTION/rdf_omp_offload_c

### **3.3 Inspecting Collapsed OpenMP Code**
Notice the `collapse(2)` clause added to the pragmas.

In [ ]:
%%bash
cd openmp/source_code

echo "--- Collapsed OpenMP Target Region ---"
grep -A 5 "#pragma omp target teams distribute" SOLUTION/rdf_offload_collapse.cpp

### **3.4 Compiling & Profiling Collapsed OpenMP Offload**

In [ ]:
%%bash
cd openmp/source_code
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Collapsed OpenMP Offload..."
nvc++ -O3 -w -I. ${NVTXLIB} -mp=gpu -gpu=cc89 -o SOLUTION/rdf_omp_collapse_c SOLUTION/rdf_offload_collapse.cpp

echo "Profiling Collapsed OpenMP Offload..."
nsys profile \
  --trace=cuda,openmp,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_omp_collapse \
  ./SOLUTION/rdf_omp_collapse_c

**💡 What to look for:** Look at the `CUDA Kernel Statistics` table above. Compare it with the Basic OpenMP version. The `collapse` version reduces the calculation time per kernel significantly because the GPU is fed much more parallel work simultaneously!

---
## **Part 4: Native CUDA - Memory Strategies**

For those writing native CUDA code, memory management is highly explicit. You can:
1. Manually allocate memory with `cudaMalloc` and move data with `cudaMemcpy`.
2. Let the NVIDIA driver handle it automatically using **Unified Virtual Memory (UVM)** via `cudaMallocManaged`.

### **4.1 Inspecting Explicit `cudaMalloc` Code**

In [ ]:
%%bash
cd cuda/source_code

echo "--- Explicit cudaMalloc Example ---"
grep -B 1 -A 5 "cudaMalloc(" SOLUTION/rdf_malloc.cu | head -n 10

### **4.2 Compiling & Profiling `cudaMalloc` Version**

In [ ]:
%%bash
cd cuda/source_code
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Explicit Malloc Version..."
nvcc -O3 -w -I. ${NVTXLIB} -o SOLUTION/rdf_malloc_c SOLUTION/rdf_malloc.cu

echo "Profiling Explicit Malloc..."
nsys profile \
  --trace=cuda,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_cuda_malloc \
  ./SOLUTION/rdf_malloc_c

### **4.3 Inspecting Unified Virtual Memory (`cudaMallocManaged`) Code**

In [ ]:
%%bash
cd cuda/source_code

echo "--- Unified Memory Example ---"
grep -B 1 -A 5 "cudaMallocManaged(" SOLUTION/rdf_unified_memory.cu | head -n 10

### **4.4 Compiling & Profiling Unified Memory Version**

In [ ]:
%%bash
cd cuda/source_code
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling Unified Memory Version..."
nvcc -O3 -w -I. ${NVTXLIB} -o SOLUTION/rdf_unified_memory_c SOLUTION/rdf_unified_memory.cu

echo "Profiling Unified Memory..."
nsys profile \
  --trace=cuda,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_cuda_um \
  ./SOLUTION/rdf_unified_memory_c

---
## **Part 5: Hardware Deep Dive with Nsight Compute (`ncu`)**

Nsight Systems (`nsys`) gives us the big picture (the forest). Now we will use Nsight Compute (`ncu`) to look closely at a single kernel (the tree).

`ncu` replays the kernel dozens of times to collect detailed hardware metrics (registers, bandwidth, warps, etc.). Because it is so detailed, profiling an entire application will take too long. Therefore, we use specific flags:
* `--launch-skip 2`: Skip the first 2 kernels (to avoid profiling un-warmed-up cache states).
* `--launch-count 1`: Only profile exactly 1 kernel execution and then let the program run normally.

### **⚠️ Note on Cloud Environments & `ERR_NVGPUCTRPERM`**
Nsight Compute requires deep hardware access. In many Docker containers and Cloud environments (like this workspace), this access is blocked by the hypervisor/host for security reasons unless the container is started with `SYS_ADMIN` capabilities. 
If you see `==ERROR== ERR_NVGPUCTRPERM` below, **do not panic!** It is entirely expected in a secured cloud container. The Jupyter cell is configured to catch this safely so we can learn from it and proceed.

In [ ]:
%%bash
cd openacc/source_code

echo "Running Detailed Hardware Analysis with Nsight Compute..."
# The '|| true' statement at the end prevents Jupyter from crashing if the cloud environment blocks hardware counters.
ncu \
  --launch-skip 2 \
  --launch-count 1 \
  --set full \
  --force-overwrite \
  -o ncu_acc_data \
  ./SOLUTION/rdf_data_c || echo -e "\n✅ [Educational Note]: Nsight Compute failed due to ERR_NVGPUCTRPERM. This confirms our cloud container restricts hardware counter access. This is expected, please proceed to Part 6!"


---
## **Part 6: Final Task - Full Profiling and Local UI Analysis**

To conclude our workshop, we will generate a comprehensive profile report containing all possible trace information. This generates an **`.nsys-rep`** file which you will download to your local machine.

Let's compile and trace the fully optimized OpenACC code (`rdf_gang_vector`) with everything turned on.

In [ ]:
%%bash
cd openacc/source_code
NVTXLIB="-I/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/include"

echo "Compiling the fully optimized gang_vector version..."
nvc++ -O3 -w -I. ${NVTXLIB} -acc -gpu=cc89 -o SOLUTION/rdf_final_c SOLUTION/rdf_gang_vector.cpp

echo "Generating comprehensive profile report (.nsys-rep)..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --sample=cpu \
  --stats=true \
  --force-overwrite=true \
  -o rdf_final_comprehensive \
  ./SOLUTION/rdf_final_c > /dev/null 2>&1
  
echo "File generated: /labs/openacc/source_code/rdf_final_comprehensive.nsys-rep"

### **🖥️ Local UI Exercise: Open and Analyze Your Profiles!**

The command line numbers are great, but visualizing the timeline is where the real power of profiling lies.

**⚠️ IMPORTANT: How to Download (Avoid UTF-8 Error)**
Because `.nsys-rep` files are binary databases, simply left-clicking on them in JupyterLab will cause a "not UTF-8 encoded" error. To download them correctly, you must use your browser's native context menu.

**Please follow these exact steps to download:**
1. Hover over the links below.
2. Hold the **Shift** key and **Right-Click** on the link to open your browser's native menu (this overrides the Jupyter menu).
3. Select **"Save Link As..."** (or "Download Linked File") to save it to your local laptop.

*(Note: The files are located relative to the `/labs` directory where this notebook is running)*

* [Download rdf_final_comprehensive.nsys-rep](openacc/source_code/rdf_final_comprehensive.nsys-rep) (Part 6: Fully Optimized OpenACC)
* [Download profile_acc_parallel.nsys-rep](openacc/source_code/profile_acc_parallel.nsys-rep) (Part 2.1: Naive Parallel OpenACC)
* [Download profile_cuda_um.nsys-rep](cuda/source_code/profile_cuda_um.nsys-rep) (Part 4.4: CUDA Unified Memory)
* [Download profile_cuda_malloc.nsys-rep](cuda/source_code/profile_cuda_malloc.nsys-rep) (Part 4.2: Explicit cudaMalloc)

---
**How to view your profiles:**
1. Open **NVIDIA Nsight Systems** on your local machine.
2. Drag and drop the downloaded `.nsys-rep` files into the Nsight Systems window.

**Checklist of what to look for in the UI:**
* 🔍 **NVTX Markers:** Look at the top rows. You will see colored bars labeled "Pair_Calculation". This helps you isolate exactly where your function is running!
* 🔍 **Compare Parallel vs Final:** Notice the overwhelming amount of green "Unified Memory" page faults or memory transfers hiding the compute blocks in early versions. Now open the `rdf_final_comprehensive` file—you should see almost entirely pure compute blocks. 
* 🔍 **Unified Memory (from Part 4):** Open `profile_cuda_um.nsys-rep` and look for "Page Faults" occurring *during* the kernel execution. Compare this to `profile_cuda_malloc.nsys-rep` where transfers happen explicitly before the kernel starts.

Congratulations on completing the profiling workshop! You now have the skills to compile, trace, measure, and visually inspect GPU performance for any scientific application.